In [1]:
!git clone https://github.com/MRoy03/Forensic_LAB.git /kaggle/working/ForensicAI_Lab


Cloning into '/kaggle/working/ForensicAI_Lab'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 85 (delta 10), reused 85 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 57.50 KiB | 8.21 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [2]:
import sys, os
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab"
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

!pip install -q -r requirements-train.txt


In [3]:
import torch, cv2, numpy as np, pandas as pd, json, shutil, zipfile
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

from config.loader import get_settings, resolve_dataset_root, resolve_path
from services.video.extractor import extract_frames, get_video_fps, get_frame_count
from services.video.sampler import sample_frame_indices
from services.preprocessing.pipeline import preprocess_image
from services.detection.model import ForensicMultiHeadNet

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device, "| dataset root:", dataset_root)


Training on: cuda | dataset root: /kaggle/working/ForensicAI_Lab/data


In [4]:
!ls /kaggle/input/


datasets


In [5]:
KAGGLE_INPUT_SLUG = "datasets"  # <-- paste the exact folder name from `!ls /kaggle/input/` above
kaggle_input_path = Path("/kaggle/input") / KAGGLE_INPUT_SLUG
zip_source_dir = kaggle_input_path if kaggle_input_path.exists() else dataset_root / "dfdc_subset"
print("Reading DFDC data from:", zip_source_dir)

zip_files = sorted(zip_source_dir.glob("**/*.zip"))
for z in zip_files:
    print(f"  {z.name}  ({z.stat().st_size / 1e9:.1f} GB)")


Reading DFDC data from: /kaggle/input/datasets


In [6]:
MAX_PARTS = 1              # scale up to len(zip_files) once this works
MAX_VIDEOS_PER_PART = 50   # scale up to None once this works
FRAMES_PER_VIDEO = 10

scratch_dir = Path("/kaggle/working/dfdc_scratch")
frames_out_dir = resolve_path("data/synthetic_data")
frames_out_dir.mkdir(parents=True, exist_ok=True)
frame_manifest_path = resolve_path("data/dfdc_frame_manifest.csv")
if frame_manifest_path.exists():
    frame_manifest_path.unlink()

sources = [(z, True) for z in zip_files] if zip_files else [(zip_source_dir, False)]
total_rows_written = 0

for part_idx, (source, needs_extract) in enumerate(sources[:MAX_PARTS]):
    print(f"\n=== Part {part_idx + 1}/{min(MAX_PARTS, len(sources))}: {source.name} ===")
    if needs_extract:
        if scratch_dir.exists():
            shutil.rmtree(scratch_dir)
        scratch_dir.mkdir(parents=True)
        with zipfile.ZipFile(source) as zf:
            zf.extractall(scratch_dir)
        search_root = scratch_dir
    else:
        search_root = source

    metadata_files = list(search_root.glob("**/metadata.json"))
    if not metadata_files:
        print(f"  No metadata.json found under {search_root}, skipping")
        if needs_extract:
            shutil.rmtree(scratch_dir)
        continue

    part_rows = []
    for meta_path in metadata_files:
        video_folder = meta_path.parent
        with open(meta_path) as f:
            metadata = json.load(f)
        items = list(metadata.items())
        if MAX_VIDEOS_PER_PART is not None:
            items = items[:MAX_VIDEOS_PER_PART]

        for filename, info in items:
            video_path = video_folder / filename
            if not video_path.exists():
                continue
            label = "real" if info["label"].upper() == "REAL" else "fake"
            manip_type = "none" if label == "real" else "gan"

            fps = get_video_fps(str(video_path))
            total = get_frame_count(str(video_path))
            indices = set(sample_frame_indices(total, fps, settings.video.sample_fps, max_frames=FRAMES_PER_VIDEO))
            for idx, frame in extract_frames(str(video_path)):
                if idx not in indices:
                    continue
                frame_name = f"{source.stem}_{video_path.stem}_frame{idx}.jpg"
                cv2.imwrite(str(frames_out_dir / frame_name), frame)
                part_rows.append({"filename": str(frames_out_dir / frame_name), "label": label, "type": manip_type})

    pd.DataFrame(part_rows, columns=["filename", "label", "type"]).to_csv(
        frame_manifest_path, mode="a", header=(total_rows_written == 0), index=False
    )
    total_rows_written += len(part_rows)
    print(f"  Extracted {len(part_rows)} rows (running total: {total_rows_written})")
    if needs_extract:
        shutil.rmtree(scratch_dir)

print(f"\nDone. {total_rows_written} total frame rows -> {frame_manifest_path}")



=== Part 1/1: datasets ===
  Extracted 5000 rows (running total: 5000)

Done. 5000 total frame rows -> /kaggle/working/ForensicAI_Lab/data/dfdc_frame_manifest.csv


In [7]:
class FrameDataset(Dataset):
    def __init__(self, manifest_csv, image_size):
        self.df = pd.read_csv(manifest_csv)
        self.image_size = image_size
        self.label_to_idx = {l: i for i, l in enumerate(settings.labels.binary)}
        self.manip_to_idx = {m: i for i, m in enumerate(settings.labels.manipulation_types)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_bgr = cv2.imread(row["filename"])
        x = preprocess_image(image_bgr, tuple(self.image_size))
        label_idx = self.label_to_idx[row["label"]]
        manip_idx = self.manip_to_idx.get(row["type"], 0)
        mask = np.zeros((1, *self.image_size), dtype=np.float32)
        return (torch.from_numpy(x), torch.tensor(label_idx), torch.tensor(manip_idx),
                torch.from_numpy(mask), torch.tensor(0.0))

frame_dataset = FrameDataset(frame_manifest_path, settings.inference.image_size)
loader = DataLoader(frame_dataset, batch_size=16, shuffle=True, num_workers=2)
print("Frame dataset size:", len(frame_dataset))

model = ForensicMultiHeadNet(
    backbone=settings.models.backbone,
    in_chans=settings.inference.input_channels,
    num_manip_classes=len(settings.labels.manipulation_types),
).to(device)

image_ckpt = resolve_path(settings.models.image_model_checkpoint)
if image_ckpt.exists():
    model.load_state_dict(torch.load(image_ckpt, map_location=device))
    print("Warm-started from Stage 1 checkpoint")
else:
    print("No Stage 1 checkpoint -- training from random weights (fine for now)")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
cls_loss_fn = torch.nn.CrossEntropyLoss()
manip_loss_fn = torch.nn.CrossEntropyLoss()
mask_loss_fn = torch.nn.BCEWithLogitsLoss(reduction="none")
NUM_EPOCHS = 5


Frame dataset size: 5000
No Stage 1 checkpoint -- training from random weights (fine for now)


In [8]:
model.train()
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    for x, label_idx, manip_idx, mask, has_mask in loader:
        x, label_idx, manip_idx = x.to(device), label_idx.to(device), manip_idx.to(device)
        mask, has_mask = mask.to(device), has_mask.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss_cls = cls_loss_fn(out["cls_logits"], label_idx)
        loss_manip = manip_loss_fn(out["manip_logits"], manip_idx)
        mask_loss_per_px = mask_loss_fn(out["mask_logits"], mask)
        loss_mask = (mask_loss_per_px.mean(dim=[1, 2, 3]) * has_mask).sum() / (has_mask.sum() + 1e-8)
        loss = loss_cls + loss_manip + loss_mask
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} -- loss: {running_loss/len(loader):.4f}")

checkpoint_path = resolve_path(settings.models.video_model_checkpoint)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), checkpoint_path)
print("Saved to", checkpoint_path, "-- download it from the Output tab and upload to your Drive.")


Epoch 1/5 -- loss: 0.7729
Epoch 2/5 -- loss: 0.4364
Epoch 3/5 -- loss: 0.2824
Epoch 4/5 -- loss: 0.1505
Epoch 5/5 -- loss: 0.0882
Saved to /kaggle/working/ForensicAI_Lab/models/video_model.pth -- download it from the Output tab and upload to your Drive.


In [9]:
# confirm it's actually there and check its size
import os
print(os.path.getsize("/kaggle/working/ForensicAI_Lab/models/video_model.pth") / 1e6, "MB")


16.050939 MB
